In [24]:
import pandas as pd
import holidays

import numpy as np
import holidays
from datetime import timedelta



# ===============================
# LOAD & PREP DATA
# ===============================
df = pd.read_csv("harga_beras_rows.csv")


def detect_iqr_outlier(df):
    df = df.copy()
    df["is_outlier"] = False

    group_cols = ["tahun", "variant_id", "tipe_harga_id", "kode_kab_kota"]

    for keys, g in df.groupby(group_cols):
        print("group is = ", group_cols)
        print("Group:", keys)
        print("Jumlah data:", len(g))

        q1 = g["harga"].quantile(0.25)
        q3 = g["harga"].quantile(0.75)
        iqr = q3 - q1

        print("Q1:", q1, "Q3:", q3, "IQR:", iqr)

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        print("Lower:", lower, "Upper:", upper)
        print("-"*40)

        idx = g[(g["harga"] < lower) | (g["harga"] > upper)].index
        df.loc[idx, "is_outlier"] = True

    return df



df["tanggal"] = pd.to_datetime(df["tanggal"])
df["tahun"] = df["tanggal"].dt.year
df = detect_iqr_outlier(df)

# ===============================
# INISIALISASI HOLIDAYS INDONESIA
# ===============================
id_holidays = holidays.Indonesia(years=range(df["tahun"].min(), df["tahun"].max() + 1))


group is =  ['tahun', 'variant_id', 'tipe_harga_id', 'kode_kab_kota']
Group: (np.int32(2021), np.int64(1), np.int64(2), np.int64(1))
Jumlah data: 291
Q1: 9810.0 Q3: 10058.5 IQR: 248.5
Lower: 9437.25 Upper: 10431.25
----------------------------------------
group is =  ['tahun', 'variant_id', 'tipe_harga_id', 'kode_kab_kota']
Group: (np.int32(2021), np.int64(1), np.int64(2), np.int64(3501))
Jumlah data: 262
Q1: 9300.0 Q3: 9300.0 IQR: 0.0
Lower: 9300.0 Upper: 9300.0
----------------------------------------
group is =  ['tahun', 'variant_id', 'tipe_harga_id', 'kode_kab_kota']
Group: (np.int32(2021), np.int64(1), np.int64(2), np.int64(3502))
Jumlah data: 196
Q1: 9600.0 Q3: 10900.0 IQR: 1300.0
Lower: 7650.0 Upper: 12850.0
----------------------------------------
group is =  ['tahun', 'variant_id', 'tipe_harga_id', 'kode_kab_kota']
Group: (np.int32(2021), np.int64(1), np.int64(2), np.int64(3503))
Jumlah data: 254
Q1: 10000.0 Q3: 10000.0 IQR: 0.0
Lower: 10000.0 Upper: 10000.0
-----------------

In [17]:
import pandas as pd

def cluster_outlier(df):
    """
    Membentuk cluster outlier berdasarkan tanggal berurutan (>1 hari putus cluster).
    
    Cluster dibuat per kombinasi:
    tahun, variant_id, tipe_harga_id
    
    Return:
        df_out  -> ringkasan cluster
        df_full -> dataframe original + kolom cluster_id
    """

    df = df.copy()

    # ==============================
    # PREPARATION
    # ==============================
    df["tanggal"] = pd.to_datetime(df["tanggal"])
    df["tahun"] = df["tanggal"].dt.year

    # ==============================
    # FILTER OUTLIER SAJA
    # ==============================
    out = df[df["is_outlier"] == True].copy()

    if out.empty:
        df["cluster_id"] = pd.NA
        return pd.DataFrame(), df

    out = out.sort_values(
        ["tahun", "variant_id", "tipe_harga_id", "tanggal"]
    )

    # ==============================
    # HITUNG SELISIH HARI
    # ==============================
    out["diff_hari"] = (
        out.groupby(
            ["tahun", "variant_id", "tipe_harga_id"]
        )["tanggal"]
        .diff()
        .dt.days
    )

    # ==============================
    # BENTUK CLUSTER LOKAL
    # ==============================
    out["cluster_local"] = (
        (out["diff_hari"] > 1) | (out["diff_hari"].isna())
    ).groupby(
        [out["tahun"], out["variant_id"], out["tipe_harga_id"]]
    ).cumsum()

    # ==============================
    # SUMMARY CLUSTER
    # ==============================
    df_out = (
        out.groupby(
            ["tahun", "variant_id", "tipe_harga_id", "cluster_local"]
        )
        .agg(
            start_date=("tanggal", "min"),
            end_date=("tanggal", "max"),
            jumlah_data=("tanggal", "count")
        )
        .reset_index()
    )

    df_out["durasi_hari"] = (
        df_out["end_date"] - df_out["start_date"]
    ).dt.days + 1

    # ==============================
    # BUAT CLUSTER_ID GLOBAL UNIK
    # ==============================
    df_out = df_out.sort_values(
        ["tahun", "variant_id", "tipe_harga_id", "start_date"]
    ).reset_index(drop=True)

    df_out["cluster_id"] = df_out.index + 1

    # ==============================
    # MAP CLUSTER_ID KE DATA OUTLIER
    # ==============================
    out = out.merge(
        df_out[
            ["tahun", "variant_id", "tipe_harga_id",
             "cluster_local", "cluster_id"]
        ],
        on=["tahun", "variant_id", "tipe_harga_id", "cluster_local"],
        how="left"
    )

    # ==============================
    # MERGE KE DATAFRAME FULL
    # ==============================
    df_full = df.merge(
        out[
            ["kode_kab_kota", "tanggal",
             "variant_id", "tipe_harga_id", "cluster_id"]
        ],
        on=["kode_kab_kota", "tanggal",
            "variant_id", "tipe_harga_id"],
        how="left"
    )

    # Gunakan tipe numerik nullable
    df_full["cluster_id"] = df_full["cluster_id"].astype("Int64")

    return df_out, df_full

df_out, df_full = cluster_outlier(df)


In [18]:
df_out.describe()

,tahun,variant_id,tipe_harga_id,cluster_local,start_date,end_date,jumlah_data,durasi_hari,cluster_id
count,581.000000,581.000000,581.000000,581.000000,581,581,581.000000,581.000000,581.000000
mean,2023.230637,1.518072,2.380379,13.807229,2023-09-18 00:02:28.709122048,2023-09-23 05:59:22.822719488,23.478485,6.247849,291.000000
min,2021.000000,1.000000,1.000000,1.000000,2021-01-04 00:00:00,2021-01-04 00:00:00,1.000000,1.000000,1.000000
25%,2022.000000,1.000000,2.000000,6.000000,2022-03-22 00:00:00,2022-03-31 00:00:00,1.000000,1.000000,146.000000
50%,2023.000000,2.000000,2.000000,12.000000,2023-12-16 00:00:00,2023-12-17 00:00:00,2.000000,2.000000,291.000000
75%,2025.000000,2.000000,3.000000,20.000000,2025-02-12 00:00:00,2025-02-22 00:00:00,11.000000,5.000000,436.000000
max,2026.000000,2.000000,3.000000,43.000000,2026-02-23 00:00:00,2026-02-26 00:00:00,877.000000,92.000000,581.000000
std,1.614105,0.500104,0.605914,9.642615,NaN,NaN,70.110245,12.482837,167.864529


In [22]:
df_full

,id,kode_kab_kota,tanggal,variant_id,harga,tipe_harga_id,tahun,is_outlier,cluster_id
0,4069,3525,2021-02-18,1,10600.0,2,2021,False,<NA>
1,4070,1,2021-02-18,1,10600.0,2,2021,True,1
2,4071,3525,2021-02-18,2,11600.0,2,2021,False,<NA>
3,4072,1,2021-02-18,2,11600.0,2,2021,True,60
4,4073,3578,2021-03-09,1,10500.0,2,2021,False,<NA>
...,...,...,...,...,...,...,...,...,...
240623,245646,3578,2026-02-26,2,15500.0,1,2026,True,564
240624,245647,3579,2026-02-26,1,12800.0,1,2026,False,<NA>
240625,245648,3579,2026-02-26,2,14750.0,1,2026,True,564
240626,245649,1,2026-02-26,1,12883.0,1,2026,True,543


In [19]:
from gnews import GNews
import urllib.parse

def get_gnews_by_date(start_date, end_date,
                      language='id', country='ID'):
    
    keyword_raw = "harga OR impor beras OR logistik OR pangan OR cuaca ekstrim jawa timur"
    keyword = urllib.parse.quote(keyword_raw)
    # ✅ Pastikan format tuple
    if not isinstance(start_date, tuple):
        start_date = (start_date.year, start_date.month, start_date.day)

    if not isinstance(end_date, tuple):
        end_date = (end_date.year, end_date.month, end_date.day)
    
    google_news = GNews(
        language=language,
        country=country
    )
    
    google_news.start_date = start_date
    google_news.end_date = end_date
    
    news = google_news.get_news(keyword)
    
    # print("DEBUG jumlah raw news:", len(news))  # tambahkan debug
    
    data = []
    for article in news:
        data.append({
            "tanggal": article.get('published date'),
            "judul": article.get('title'),
            "sumber": article.get('publisher', {}).get('title'),
            "url": article.get('url')
        })
    
    df_news = pd.DataFrame(data)

    kalimat = df_to_sentences(df_news)
    
    # print("Jumlah berita: \n", kalimat)
    
    return kalimat

def df_to_sentences(df):
    sentences = []
    
    for _, row in df.iterrows():
        kalimat = (
            f"Pada {row['tanggal']}, media {row['sumber']} "
            f"memberitakan: \"{row['judul']}\"."
        )
        sentences.append(kalimat)
    
    return sentences

# ===============================
# FUNGSI UNTUK DETEKSI HARI LIBUR DI RENTANG
# ===============================
def get_holidays_in_range(start_date, end_date, buffer_days=7):
    """
    Deteksi hari libur dalam rentang tanggal, plus buffer sebelum/sesudah
    """
    holiday_list = []
    
    # Extend range dengan buffer
    extended_start = start_date - timedelta(days=buffer_days)
    extended_end = end_date + timedelta(days=buffer_days)
    
    current = extended_start
    while current <= extended_end:
        if current in id_holidays:
            holiday_list.append({
                'date': current,
                'name': id_holidays[current]
            })
        current += timedelta(days=1)
    
    return holiday_list



import re
from datetime import datetime

def format_news_item(news_text):
    """
    Parsing string berita menjadi format:
    - Menurut media X, pada Tanggal, Judul
    """

    # Contoh format:
    # Pada Sat, 10 Apr 2021 07:00:00 GMT, media Ngopibareng.id memberitakan: "Judul - Ngopibareng.id"

    pattern = r"Pada .*?, (\d{2} \w{3} \d{4}).*?media (.*?) memberitakan: \"(.*?)\""
    match = re.search(pattern, news_text)

    if match:
        tanggal = match.group(1)
        media = match.group(2)
        judul = match.group(3).split(" - ")[0]  # buang suffix nama media

        return f"- Menurut media {media}, pada {tanggal}, {judul}"
    
    return None


def ask_event_indonesia(start_date, end_date):

    holidays_found = get_holidays_in_range(start_date, end_date, buffer_days=7)
    berita = get_gnews_by_date(start_date, end_date)
    # print(berita)
    output_lines = []

    # ===============================
    # HOLIDAY SECTION
    # ===============================
    if holidays_found:
        output_lines.append("Berdekatan dengan hari besar:")

        for h in holidays_found:
            tanggal = h['date'].strftime('%d-%m-%Y')
            nama = h['name']
            output_lines.append(f"- {nama} ({tanggal})")


    # ===============================
    # NEWS SECTION
    # ===============================
    formatted_news = []


    for item in berita:
        formatted = format_news_item(item)
        if formatted:
            formatted_news.append(formatted)

    if formatted_news:
        if output_lines:
            output_lines.append("")  # spasi antar section
        output_lines.append("Terjadi peristiwa:")
        output_lines.extend(formatted_news)

    



    print(f"total berita {len(formatted_news)} | holiday {len(output_lines)}")
    print(f"hari besar \n {output_lines}")
    print(f"news \n {formatted_news}")

    # ===============================
    # FALLBACK
    # ===============================
    if not output_lines:
        return "Tidak ditemukan peristiwa dalam periode tersebut."

    return "\n".join(output_lines)


In [20]:
import time

def analyze_df_events(df_group, df_detail, sample_n=None, delay=2, error_delay=5):
    """
    Tambahkan kolom 'deskripsi' berdasarkan start_date & end_date.

    Parameters:
        df_group   : dataframe cluster (start_date, end_date, cluster_id)
        df_detail  : dataframe detail (is_outlier, id, cluster_id)
        sample_n   : jumlah baris diproses (None = semua)
        delay      : jeda normal antar request (detik)
        error_delay: jeda jika error (detik)
    """

    if sample_n == 999:
        sample_n = len(df_detail[df_detail["is_outlier"] == True])

    df = df_group.copy()

    df['start_date'] = pd.to_datetime(df['start_date'])
    df['end_date']   = pd.to_datetime(df['end_date'])

    target_df = df.head(sample_n) if sample_n else df

    total = len(target_df)

    print(f"\nTotal yang diproses: {total} baris\n")

    for i, (idx, row) in enumerate(target_df.iterrows(), 1):

        print(f" id {idx} [{i}/{total}] {row['start_date'].date()} → {row['end_date'].date()} ...", end=" ")

        try:
            analisis = ask_event_indonesia(row['start_date'], row['end_date'])
            df.loc[idx, 'deskripsi'] = analisis
            # print("✅")

            # delay normal
            if i < total:
                time.sleep(delay)

        except Exception as e:
            df.loc[idx, 'deskripsi'] = f"ERROR: {e}"
            print(f"❌ {e}")

            # delay lebih lama jika error
            time.sleep(error_delay)

    # ===============================
    # PREPARE OUTPUT
    # ===============================

    # ===============================
    # OUTLIER DETAIL
    # ===============================
    df_outlier_detail = (
        df_detail[df_detail["is_outlier"] == True][['id', 'cluster_id']]
        .reset_index(drop=True)
    )

    df_outlier_detail.insert(0, 'row_id', range(1, len(df_outlier_detail) + 1))


    # ===============================
    # OUTLIER GROUP
    # ===============================
    df_outlier_group = (
        df[['cluster_id', 'deskripsi']]
        .reset_index(drop=True)
    )

    df_outlier_group.insert(0, 'row_id', range(1, len(df_outlier_group) + 1))

    df_outlier_detail = df_outlier_detail.reset_index(drop=True)
    df_outlier_group  = df_outlier_group.reset_index(drop=True)

    return df_outlier_detail, df_outlier_group


# # ===============================
# # CONTOH PENGGUNAAN
# # ===============================

# # Test 3 baris dulu sebelum proses semua
# outlier_detail, outlier_group = analyze_df_events(df_out, df_full, sample_n=1)
# # print(outlier_group.iloc[0]['deskripsi'])

In [21]:
def analyze_in_batches(df_group, df_detail, batch_size=50, delay_between_batch=10):
    
    total = len(df_group)
    results_detail = []
    results_group = []
    
    print(f"Total cluster: {total}")
    print(f"Batch size: {batch_size}")
    print(f"Total batch: {(total // batch_size) + 1}\n")
    
    for start in range(0, total, batch_size):
        
        end = start + batch_size
        print(f"\n===== PROSES BATCH {start} - {min(end, total)} =====")
        
        df_batch = df_group.iloc[start:end]
        
        detail, group = analyze_df_events(
            df_batch,
            df_detail,
            sample_n=None,
            delay=2,
            error_delay=5
        )
        
        results_detail.append(detail)
        results_group.append(group)
        
        # Delay antar batch supaya API aman
        if end < total:
            print(f"\n⏳ Tunggu {delay_between_batch} detik sebelum batch berikutnya...\n")
            time.sleep(delay_between_batch)
    
    final_detail = pd.concat(results_detail, ignore_index=True)
    final_group  = pd.concat(results_group, ignore_index=True)
    
    return final_detail, final_group

outlier_detail, outlier_group = analyze_in_batches(
    df_out,
    df_full,
    batch_size=50,
    delay_between_batch=15
)

Total cluster: 581
Batch size: 50
Total batch: 12


===== PROSES BATCH 0 - 50 =====

Total yang diproses: 50 baris

 id 0 [1/50] 2021-02-18 → 2021-02-18 ... 

/home/dimas/.local/lib/python3.10/site-packages/gnews/gnews.py:146: UserWarning: The start and end dates should be at least 1 day apart.
  warnings.warn("The start and end dates should be at least 1 day apart.")


total berita 0 | holiday 2
hari besar 
 ['Berdekatan dengan hari besar:', '- Lunar New Year (12-02-2021)']
news 
 []
 id 1 [2/50] 2021-03-09 → 2021-03-10 ... total berita 1 | holiday 6
hari besar 
 ['Berdekatan dengan hari besar:', "- Isra' and Mi'raj (11-03-2021)", '- Day of Silence (14-03-2021)', '', 'Terjadi peristiwa:', '- Menurut media Tabloid Sinar Tani, pada 09 Mar 2021, Harga Cabai Rawit Kembali bikin Pedas Konsumen']
news 
 ['- Menurut media Tabloid Sinar Tani, pada 09 Mar 2021, Harga Cabai Rawit Kembali bikin Pedas Konsumen']
 id 2 [3/50] 2021-03-15 → 2021-04-10 ... total berita 4 | holiday 10
hari besar 
 ['Berdekatan dengan hari besar:', "- Isra' and Mi'raj (11-03-2021)", '- Day of Silence (14-03-2021)', '- Good Friday (02-04-2021)', '', 'Terjadi peristiwa:', '- Menurut media Ngopibareng.id, pada 10 Apr 2021, Jatim Masuki Musim Pancaroba, BPBD: Waspadai Angin Puting Beliung', '- Menurut media Ruang Energi, pada 05 Apr 2021, Pertamina Antisipasi Dampak Cuaca Ekstrim di NTT »

/home/dimas/.local/lib/python3.10/site-packages/gnews/gnews.py:146: UserWarning: The start and end dates should be at least 1 day apart.
  warnings.warn("The start and end dates should be at least 1 day apart.")


total berita 0 | holiday 3
hari besar 
 ['Berdekatan dengan hari besar:', '- Ascension Day; Eid al-Fitr (13-05-2021)', '- Eid al-Fitr Second Day (14-05-2021)']
news 
 []
 id 7 [8/50] 2021-05-17 → 2021-05-22 ... total berita 0 | holiday 4
hari besar 
 ['Berdekatan dengan hari besar:', '- Ascension Day; Eid al-Fitr (13-05-2021)', '- Eid al-Fitr Second Day (14-05-2021)', '- Vesak Day (26-05-2021)']
news 
 []
 id 8 [9/50] 2021-05-24 → 2021-05-27 ... total berita 0 | holiday 3
hari besar 
 ['Berdekatan dengan hari besar:', '- Vesak Day (26-05-2021)', '- Pancasila Day (01-06-2021)']
news 
 []
 id 9 [10/50] 2021-05-30 → 2021-05-30 ... 

/home/dimas/.local/lib/python3.10/site-packages/gnews/gnews.py:146: UserWarning: The start and end dates should be at least 1 day apart.
  warnings.warn("The start and end dates should be at least 1 day apart.")


total berita 0 | holiday 3
hari besar 
 ['Berdekatan dengan hari besar:', '- Vesak Day (26-05-2021)', '- Pancasila Day (01-06-2021)']
news 
 []
 id 10 [11/50] 2021-06-03 → 2021-06-03 ... 

/home/dimas/.local/lib/python3.10/site-packages/gnews/gnews.py:146: UserWarning: The start and end dates should be at least 1 day apart.
  warnings.warn("The start and end dates should be at least 1 day apart.")


total berita 0 | holiday 2
hari besar 
 ['Berdekatan dengan hari besar:', '- Pancasila Day (01-06-2021)']
news 
 []


KeyboardInterrupt: 

In [ ]:
outlier_detail.to_csv("detail_out.csv")

In [ ]:
outlier_group.to_csv("group_out.csv")